# Assignment: Sentiment Analysis Chatbot

**- Name:** Avnish Agrawal | 
**- Reg No:** 23BAI10628

---

Develop a robust, sentiment-aware conversational agent. This project utilizes Natural Language Processing (NLP) to train a classification model on the Sentiment140 dataset, combined with an interactive web user interface built via Streamlit.

### Project Summary
This notebook provides an end-to-end workflow for detecting emotional polarity in text. It processes a 100,000-tweet sample from the Sentiment140 dataset to optimize speed, standardizes binary positive/negative labels, and extracts features using a `TfidfVectorizer` (up to 10,000 features). A **Logistic Regression** classifier is then trained and deployed through an interactive web-based chat interface.

## 1. Environment Setup
Installing the necessary dependencies required for data manipulation (`pandas`), machine learning (`scikit-learn`), and the web interface (`streamlit`).

In [1]:
# Install required libraries
# !pip install -q streamlit pandas scikit-learn


## 2. Application Backend and User Interface
Because Streamlit applications require a standalone Python script to execute, the following cell writes the complete application code to a file named `sentiment_chatbot.py`.

**Key Pipeline Components:**
*   **Data Preprocessing**: Cleans the raw tweet text using regular expressions to remove URLs, mentions, and hashtags. Converts all text to lowercase.
*   **Model Training**: Trains the Logistic Regression model. The `@st.cache_resource` decorator ensures the model only trains once per session, preventing lag during the chat.
*   **Web Interface**: Initializes a `session_state` to store and display the ongoing conversation history. It accepts real-time user input, passes it through the trained model, and returns a dynamically generated bot response with a confidence score.

In [2]:
%%writefile sentiment_chatbot.py
import streamlit as st
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# --- 1. MODEL TRAINING PIPELINE ---
# The @st.cache_resource decorator ensures the model trains only once per session
@st.cache_resource
def train_sentiment_model():
    columns = ['target', 'ids', 'date', 'flag', 'user', 'text']
    
    # Load dataset - using a 100k sample for optimal web app performance
    # Note: Ensure 'training.1600000.processed.noemoticon.csv' is in the root directory
    df = pd.read_csv('training.1600000.processed.noemoticon.csv', 
                     encoding='latin-1', names=columns).sample(100000, random_state=42)
    
    df = df[['target', 'text']]
    df['target'] = df['target'].replace(4, 1)
    
    def clean_text(text):
        text = re.sub(r"http\S+|www\S+|https\S+", '', text, flags=re.MULTILINE)
        text = re.sub(r'\@\w+|\#', '', text)
        return text.lower()
    
    df['text'] = df['text'].apply(clean_text)
    
    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['target'], test_size=0.2, random_state=42)
    
    vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
    X_train_vec = vectorizer.fit_transform(X_train)
    
    model = LogisticRegression(max_iter=500)
    model.fit(X_train_vec, y_train)
    
    return model, vectorizer

# --- 2. STREAMLIT UI SETUP ---
st.set_page_config(page_title="SentimentBot", page_icon="🤖", layout="centered")

st.title("🤖 NLP Sentiment Analysis Chatbot")
st.markdown("**Developed by: PRAKHAR BHARDWAJ (23MEI10051)**")
st.markdown("This chatbot utilizes a Logistic Regression model trained on the **Sentiment140** Twitter dataset to classify your input as Positive or Negative.")
st.divider()

with st.spinner("Initializing Model & Processing Dataset..."):
    model, vectorizer = train_sentiment_model()

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- 3. INTERACTIVE CHAT LOGIC ---
if user_input := st.chat_input("Enter your message here..."):
    
    with st.chat_message("user"):
        st.markdown(user_input)
    
    st.session_state.messages.append({"role": "user", "content": user_input})
    
    input_vec = vectorizer.transform([user_input])
    prediction = model.predict(input_vec)[0]
    confidence = model.predict_proba(input_vec)[0].max() * 100
    
    if prediction == 1:
        bot_response = f"**Positive Sentiment Detected!** ({confidence:.1f}% confidence) \n\nI'm glad to hear that! Keep spreading the good vibes! ✨"
    else:
        bot_response = f"**Negative Sentiment Detected...** ({confidence:.1f}% confidence) \n\nI'm sorry to hear that. I hope your day gets better! 💙"
        
    with st.chat_message("assistant"):
        st.markdown(bot_response)
        
    st.session_state.messages.append({"role": "assistant", "content": bot_response})


Writing sentiment_chatbot.py


## 3. Launching the Application
Running the cell below will initialize the Streamlit server. Once started, the application can be accessed via your local browser at `http://localhost:8501`.

In [3]:
# Execute the Streamlit web application
# !streamlit run sentiment_chatbot.py
